# 다중공선성 분석

1. 상관관계 히트맵
2. VIF (Variance Inflation Factor)
3. 제거 대상 변수 확정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

BASE = Path('..').resolve()
df_raw = pd.read_csv(
    BASE / 'data/02_interim/260506_feature_engineering/promotion_0_membership_v3.csv',
    encoding='utf-8-sig'
)
print(f'원본 shape: {df_raw.shape}')

In [ ]:
# 분석에서 제외할 컬럼 (ID, 날짜, 문자열, 타겟, 이미 제거 결정)
DROP_COLS = [
    'USER_KEY', 'product_code', 'payment_device', 'device_group',
    'gender', 'reg_date', 'end_date', 'age_group', 'price_tier',
    # 제거 결정된 변수
    'price_per_screen', 'is_new_product', 'is_apple_ecosystem',
    'weekday_watch_ratio', 'is_long_sub', 'hour_x_weekend',
    'verified_x_age',
    # 타겟
    'is_repurchase',
]
DROP_COLS = [c for c in DROP_COLS if c in df_raw.columns]

df = df_raw.drop(columns=DROP_COLS)

# 수치형만 선택
df_num = df.select_dtypes(include=[np.number]).fillna(0)
print(f'분석 대상 변수: {df_num.shape[1]}개')
print(list(df_num.columns))

## 1. 상관관계 히트맵

In [ ]:
corr = df_num.corr()

fig, ax = plt.subplots(figsize=(20, 18))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=False, cmap='coolwarm',
    center=0, vmin=-1, vmax=1,
    linewidths=0.3, ax=ax
)
ax.set_title('변수 간 상관관계 히트맵', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 절댓값 상관계수 0.7 이상 쌍 출력
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        val = corr.iloc[i, j]
        if abs(val) >= 0.7:
            high_corr.append({
                '변수1': corr.columns[i],
                '변수2': corr.columns[j],
                '상관계수': round(val, 3)
            })

high_corr_df = pd.DataFrame(high_corr).sort_values('상관계수', key=abs, ascending=False)
print(f'상관계수 |r| >= 0.7 쌍: {len(high_corr_df)}개')
high_corr_df

## 2. VIF (분산팽창계수)

- VIF < 5: 문제 없음
- VIF 5~10: 주의
- VIF > 10: 심각한 다중공선성

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 상수항 추가
X = df_num.copy()
X.insert(0, 'const', 1)

vif_data = pd.DataFrame()
vif_data['변수'] = X.columns[1:]  # const 제외
vif_data['VIF'] = [
    variance_inflation_factor(X.values, i+1)
    for i in range(len(X.columns)-1)
]
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)
vif_data['위험도'] = vif_data['VIF'].apply(
    lambda v: '🔴 심각' if v > 10 else ('🟡 주의' if v > 5 else '🟢 양호')
)

print('=== VIF 결과 ===')  
print(vif_data.to_string())

In [ ]:
# VIF 시각화
fig, ax = plt.subplots(figsize=(10, 12))
colors = ['#e74c3c' if v > 10 else '#f39c12' if v > 5 else '#2ecc71'
          for v in vif_data['VIF']]
ax.barh(vif_data['변수'][::-1], vif_data['VIF'][::-1], color=colors[::-1])
ax.axvline(x=5,  color='orange', linestyle='--', label='VIF=5 (주의)')
ax.axvline(x=10, color='red',    linestyle='--', label='VIF=10 (심각)')
ax.set_title('VIF (분산팽창계수)', fontsize=13)
ax.set_xlabel('VIF')
ax.legend()
plt.tight_layout()
plt.show()

## 3. 제거 검토 대상 정리

In [ ]:
print('=== VIF > 10 (심각) ===')  
print(vif_data[vif_data['VIF'] > 10][['변수','VIF']].to_string())

print()
print('=== VIF 5~10 (주의) ===')
print(vif_data[(vif_data['VIF'] > 5) & (vif_data['VIF'] <= 10)][['변수','VIF']].to_string())

print()
print('참고: XGBoost는 다중공선성에 강하므로 VIF > 10이어도 SHAP 분석 후 최종 결정')